In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read,write
from ase.build.surface import mx2
from ase.calculators.espresso import Espresso
from ase.io import read,write
from os import environ
from mace.calculators import MACECalculator
from os import chdir

a_optB88_QE = {}
a_optB88_QE[1,0] = 3.190902132
a_optB88_QE[0,0] = 3.186305534
a_optB88_QE[1,1] = 3.322874304
a_optB88_QE[0,1] = 3.322339468

t_optB88_QE = {}
t_optB88_QE[1,0] = 3.1570497014
t_optB88_QE[0,0] = 3.140170013
t_optB88_QE[1,1] = 3.367178811
t_optB88_QE[0,1] = 3.3492559072

#Load MACE model
model_path = '../training/MoWSSe_bi_radius_8_reduced_all_1_swa_800_run-123_swa.model'
CALC = MACECalculator(model_paths=model_path, device="cuda", default_dtype="float64")

# define environment variable and pseudo directory for Espresso calculations
environ["ASE_ESPRESSO_COMMAND"]="mpirun -n 24 /storage/nanosim/qe-7.0/bin/pw.x -in PREFIX.pwi >> PREFIX.pwo"
pseudo_dir = '/home/theory/phrrpq/GBRV_PBE/'

def singlepoint(model,label,func,cutoff,kpts='gamma',pseudos=None,metal=False,spin=0,
                forces=True,stresses=False,writeonly=False,readonly=False,
                calconly=False,mdtemp=None):

    input_data = {
        'control': {
             'pseudo_dir': pseudo_dir,
             'prefix': label,
             'restart_mode': 'from_scratch',
             'tprnfor': forces,
             'tstress': stresses},
        'system': {
             'ecutwfc': cutoff,
             'ecutrho': float(cutoff)*9,
             'ibrav'  : 0,
             'input_dft': func}, 
        'electrons': {
             'startingpot': 'file',
             'startingwfc': 'atomic',
             'conv_thr': 1e-9,
             'mixing_beta': 0.5},
        'disk_io': 'low'} 

    if metal:
        input_data['system']['occupations'] = 'smearing'
        input_data['system']['smearing'] = 'gaussian'
        input_data['system']['degauss'] = 0.0005
    if spin != 0:
        input_data['system']['tot_magnetization'] = spin
        input_data['system']['nspin'] = 2
    calculation = 'scf'

    calc = Espresso(label=label,pseudo_dir=pseudo_dir,calculation=calculation, kpts=kpts,
             tstress=stresses,tprnfor=forces,pseudopotentials=pseudos,xc=func,
             input_data=input_data)

    model.set_calculator(calc)
    if writeonly:
        if hasattr(calc,'template'):
            from ase.io.espresso import write_espresso_in
            write_espresso_in(open(f"{label}.pwi","w"),model,input_data,pseudos,kpts=kpts)
        else:
            calc.write_input(model)
        return
    if readonly:
        if hasattr(calc,'template'):
            calc.template.read_results(calc.directory)
        else:
            calc.read_results(model)
        energy = calc.get_property('energy',atoms=model,allow_calculation=False)
        if forces:
            forces = calc.get_property('forces',atoms=model,allow_calculation=False)
    else:
        energy = model.get_potential_energy()
        if forces:
            forces = model.get_forces()
    return energy, forces

from ase.io import Trajectory

pseudos = {'Mo': 'mo_pbe_v1.uspp.F.UPF','S': 's_pbe_v1.4.uspp.F.UPF','W':'w_pbe_v1.2.uspp.F.UPF','Se':'se_pbe_v1.uspp.F.UPF'}
comps = {'MoS2':(0,0),'WS2':(1,0),'MoSe2':(0,1),'WSe2':(1,1)}

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


# Make Espresso input files for strained configurations

In [3]:
chdir('espresso_strain/')

for m1 in ['Mo','W']:
    for x1 in ['S','Se']:
        
        x,y = comps[f'{m1}{x1}2']
        orig_atoms = mx2(formula=f'{m1}{x1}2',kind='2H',a=a_optB88_QE[x,y],thickness=t_optB88_QE[x,y],vacuum=8,size=(1,1,1))
        alpha = 0.02
        espresso_range=np.arange(0.98,1.02,0.01)

        for s2 in 1 - np.array([-1, 0, 1]) * alpha:
            for (l1, l2) in ((0,0), (1,0), (1,1)):
                for s1 in espresso_range:
                    mat = np.array([[s1, l1*alpha, 0],
                                    [l2*alpha, s2, 0],
                                    [0, 0, 1]])
                    scaled_cell = mat.dot(orig_atoms.cell)
                    scaled_atoms = orig_atoms.copy()
                    scaled_atoms.set_cell(scaled_cell, scale_atoms=True)

                    filename = f'{m1}{x1}2_s1_{s1}_s2_{s2}_l1_{l1}_l2_{l2}.pwi'
                    singlepoint(scaled_atoms, filename,
                                'VDW-DF-OBK8', 50, kpts=(12,12,1),
                                pseudos=pseudos, metal=False, stresses=True,
                                writeonly=True)

chdir('../')

# Compare DFT vs MLIP energies for strained unit cells of pure monolayers

In [ ]:
from matplotlib.table import Table
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
fontsize =8
plt.rcParams['font.size'] = fontsize
plt.rcParams["figure.figsize"] = (9.5,10)

fig,axs = plt.subplots(2,2)
alpha=0.02
s1_range=np.arange(0.98,1.02,0.001)
espresso_range=np.arange(0.98,1.02,0.01)
chr_index = 0

# Loop over the pure monolayers
for m1 in ['Mo','W']:
    for x1 in ['S','Se']:
        
        x,y = comps[f'{m1}{x1}2']
        
        #Make unstrained unit cell
        orig_atoms = mx2(formula=f'{m1}{x1}2',kind='2H',a=a_optB88_QE[x,y],
                         thickness=t_optB88_QE[x,y],vacuum=8,size=(1,1,1))
        #List to store errors in energies between DFT and MLIP
        errors = []
        
        # List to store table content for strain legend
        cell_text = [[r'$\bf{\sigma_{2}}$',r'$\bf{\lambda_{1}}$',r'$\bf{\lambda_{2}}$']]
        symbol_colors = []
        
        #
        for s2 in 1-np.array([-1,0,1])*alpha:
            for (l1,l2) in ((0,0),(1,0),(1,1)):
                energies = np.zeros(len(s1_range))
                espresso_energies = np.zeros(len(espresso_range))
                k=0
                for i,s1 in enumerate(s1_range):
                    
                    #Scale original cell and calculate MLIP energy
                    mat = np.array([[s1,l1*alpha,0],[l2*alpha,s2,0],[0,0,1]])
                    scaled_cell = mat.dot(orig_atoms.cell)
                    scaled_atoms = orig_atoms.copy()
                    scaled_atoms.set_cell(scaled_cell,scale_atoms=True)
                    scaled_atoms.calc = CALC
                    energies[i] = scaled_atoms.get_potential_energy()
                    # Get DFT energy if in Espresso range
                    if s1 in espresso_range:
                        filename = f'{m1}{x1}2_s1_{s1}_s2_{s2}_l1_{l1}_l2_{l2}.pwi.pwo'
                        espresso_out = read(f'espresso_strain/{filename}')
                        espresso_energies[k] = espresso_out.get_potential_energy()
                        errors.append(abs(espresso_energies[k]-energies[i]))
                        k+=1
                
                #Plot MLIP data as line
                line = axs[x][y].plot(s1_range,energies,
                                label=rf'{s2},{l1*alpha},{l2*alpha}',lw=0.5)
                # Plot DFT data as scatter plot
                scatter = axs[x][y].scatter(espresso_range,espresso_energies,edgecolors=line[0].get_color(),
                                      marker='o',facecolor='none')
                #Append legend line color for strain and table text
                symbol_colors.append(line[0].get_color())
                cell_text.append([f'{s2:.2f}',f'{l1*alpha:.2f}',f'{l2*alpha:.2f}'])
        
        #Set up table for strain legend ------
        #Set table location, length and width
        table_xmin = 0.45
        table_ymin = 0.65
        table_height = 0.3
        table_width = 0.3

        table = axs[x][y].table(cellText=cell_text,#
                          bbox=[table_xmin,table_ymin,table_width,table_height],
                          cellLoc='center')

        table.set_fontsize(fontsize-0.7)

        for key, cell in table.get_celld().items():
            cell.set_linewidth(0.05)

        table_caption = f'MAE (in eV) : {sum(errors)/len(errors):.4f}'

        axs[x][y].text(table_xmin+0.5*table_width,table_ymin+table_height+0.01,
                 table_caption,transform=axs[x][y].transAxes,
                 ha='center',va='bottom',weight='bold')  

        linepatch_length = 0.04
        table.properties()['celld'][0,0].get_center()
        
        for i in range(len(cell_text)-1):
            x_center,y_center = table.properties()['celld'][i+1,0].get_center()
            linepatch_start = x_center-0.11
            line = Line2D([linepatch_start,linepatch_start+linepatch_length], [y_center,y_center],
                          linewidth=1,color=symbol_colors[i],transform=axs[x][y].transAxes)
        

            axs[x][y].add_line(line)
        #--------------------------------
        
        axs[x][y].text(0.3,0.98,rf"{m1}{x1}$\bf _2$",
               transform=axs[x][y].transAxes,fontsize=fontsize+4,ha='right',va='top',weight='bold')
        
        if x!=1:
            axs[x][y].tick_params(labelbottom=False)
        
        axs[x][y].xaxis.set_minor_locator(MultipleLocator(0.001))
        axs[x][y].text(-0.06,1,f"({chr(ord('a')+chr_index)})",transform=axs[x][y].transAxes,
                       ha='center',va='bottom',fontsize=fontsize+4,weight='bold')

        chr_index+=1

# --- Final layout tweaks ---
plt.subplots_adjust(wspace=0.2, hspace=0.1)                 
fig.text(0.5, 0.08, r'$\sigma_{1}$', ha='center', fontsize=fontsize+4)
fig.text(0.06, 0.5,r'E($\sigma_{1}$;$\sigma_{2}$,$\lambda_{1}$,$\lambda_{2}$) [in eV]', va='center',
         rotation='vertical',fontsize=fontsize+4)

plt.savefig('strain_test.png',dpi=300,facecolor='white',transparent=False,
            bbox_inches='tight',pad_inches=0.02)
plt.show()